## Camada 3 - o laço do agente, escrito à mão

Todo framework de agentes por aí - LangChain, n8n, Claude Code,
OpenCode - tem este laço no núcleo: o modelo decide entre acionar
uma ferramenta e encerrar com a resposta; a ferramenta executa e
devolve a observação; repete até `answer` ou o teto de passos.

Detalhe de arquitetura: a ação vai por JSON no prompt, em vez do
*function calling* nativo do provedor. O conceito é o mesmo e
funciona com qualquer modelo - inclusive os free e menores.
O padrão é sempre este: decidir, agir, observar, decidir de novo.

> **Aviso:** `exec_python` executa código gerado pelo modelo no seu
> processo, sem sandbox. Rode em ambiente descartável e não deixe dados
> sensíveis ao alcance.


In [1]:
# Setup: dados, credenciais e o helper agnóstico (llm.py)
import os, pathlib, sys
sys.path.append(str(pathlib.Path.cwd().parent))

import pandas as pd
from llm import chat, last_usage, usage_total

print(f"Modelo: {os.environ.get('LLM_MODEL', '?')}")

df = pd.read_csv('../data/reclamacoes.csv')
df.head(3)


Modelo: deepseek-v4.1-flash


,texto,categoria
0,O rastreador do pedido não atualiza há duas se...,entrega
1,Cobraram o frete duas vezes na minha fatura do...,cobranca
2,O produto chegou com a embalagem amassada e se...,produto


In [2]:
# Três ferramentas com papéis distintos:
#   load_csv: entrada de dados (a base entra na conversa)
#   exec_python: computação (o agente programa e lê o que imprimiu)
#   wiki_busca: conhecimento externo (sem chave, sem custo, sem mágica)
# A extensibilidade do padrão é o ponto central: meia dúzia de linhas
# e o agente ganha uma habilidade nova, sem re-treinar nada
import contextlib, io, json, urllib.parse, urllib.request

def load_csv() -> str:
    return df.sample(5).to_string() + f'\n\nTotal de {len(df)} linhas; colunas: {list(df.columns)}'

def exec_python(codigo: str) -> str:
    # AVISO: exec roda código gerado pelo modelo no seu processo, sem
    # sandbox. Use ambiente descartável e não passe dados sensíveis.
    saida = io.StringIO()
    # namespace fresco a cada chamada: nada do que o codigo do agente
    # redefinir (pd, df) vaza para o proximo turno
    try:
        with contextlib.redirect_stdout(saida):
            exec(codigo, {'pd': pd, 'df': df})
        
        resultado = saida.getvalue()
    
    except Exception as erro:
        resultado = f'ERRO: {erro}'
    
    return resultado[-3000:] or '(sem saída impressa; lembre-se: PRECISA usar print())'

def wiki_busca(termo: str) -> str:
    """Busca na Wikipédia (gratuita, sem chave). Exemplo de extensão:
    algumas linhas e o agente passou a consultar o mundo externo."""
    params = {'action': 'opensearch', 'format': 'json', 'search': termo, 'limit': 3}
    url = 'https://pt.wikipedia.org/w/api.php?' + urllib.parse.urlencode(params)
    req = urllib.request.Request(url, headers={'User-Agent': 'ia-hands-on-agents/1.0'})
    
    try:
        with urllib.request.urlopen(req, timeout=15) as resposta:
            titulos, descricoes, links = json.loads(resposta.read().decode())[1:4]
    
    except Exception as erro:
        return f'ERRO na busca: {erro}'
    
    if not titulos:
        return '(nenhum resultado)'
    
    return '\n'.join(f'- {t}: {d} ({l})' for t, d, l in zip(titulos, descricoes, links))

TOOLS = {'load_csv': load_csv, 'exec_python': exec_python, 'wiki_busca': wiki_busca}


In [3]:
# O system prompt define identidade, cataloga as ferramentas e amarra
# o formato estrito da resposta. A 'inteligência' de escolher entre
# tool e answer nasce daqui - e do próprio modelo.
SYSTEM = '\n'.join([
    'Voce e um agente analista de dados. Para responder, use as ferramentas.',
    'Ferramentas:',
    '- load_csv(): amostra de reclamacoes.csv (colunas: texto, categoria)',
    '- exec_python(codigo): roda Python com pandas; o DataFrame esta em df; PRECISA usar print()',
    '- wiki_busca(termo): busca na Wikipedia (use quando o usuario pedir referencia externa)',
    '',
    'Em cada turno, responda SOMENTE um JSON, sem texto ao redor:',
    '  {"tool": "load_csv", "args": {}}',
    '  {"tool": "exec_python", "args": {"codigo": "...um passo apenas..."}}',
    '  {"tool": "wiki_busca", "args": {"termo": "..."}}',
    '  {"answer": "..."}  quando tiver a resposta final para o usuario.',
])

# Extração tolerante de JSON: modelos reais às vezes embrulham o JSON
# em texto; pegar do primeiro '{' até o último '}' cobre o caso comum
def parse_json(texto: str) -> dict | None:    
    a, b = texto.find('{'), texto.rfind('}')
    
    if a < 0 or b <= a:
        return None
    
    try:
        return json.loads(texto[a:b + 1])
    except json.JSONDecodeError:
        return None


In [4]:
# O laço do agente, sem framework: cada volta é um 'turno' do modelo.
# max_steps existe por segurança: sem ele, um agente perdido gira para sempre
def run_agent(pergunta: str, max_steps: int = 6):
    mensagens = [
        {'role': 'system', 'content': SYSTEM},
        {'role': 'user', 'content': pergunta},
    ]

    indice_arg = {'exec_python': 'codigo', 'wiki_busca': 'termo'}
    
    for passo in range(max_steps):
        bruto = chat(mensagens, temperature=0, max_tokens=800)
    
        uso = last_usage()
    
        print(f'--- passo {passo}: {bruto[:140]}')
    
        if uso:
            print(f'    tokens: {uso["total_tokens"]} (prompt {uso["prompt_tokens"]}, completion {uso["completion_tokens"]})')
        acao = parse_json(bruto)
    
        if acao is None:
            mensagens += [
                {'role': 'assistant', 'content': bruto},
                {'role': 'user', 'content': 'Responda apenas com um JSON no formato definido.'},
            ]
            continue
    
        if not isinstance(acao, dict):
            mensagens += [
                {'role': 'assistant', 'content': bruto},
                {'role': 'user', 'content': 'Responda com um objeto JSON ({"tool": ...} ou {"answer": ...}).'},
            ]
            continue
    
        if 'answer' in acao:
            return acao['answer']
        nome = acao.get('tool')
    
        if not isinstance(nome, str) or nome not in TOOLS:
            mensagens += [
                {'role': 'assistant', 'content': bruto},
                {'role': 'user', 'content': f'Ferramenta desconhecida: {nome!r}. Use load_csv, exec_python ou wiki_busca.'},
            ]
            continue
        args = acao.get('args') or {}  # cobre args ausentes E args: null
    
        if not isinstance(args, dict):
            mensagens += [
                {'role': 'assistant', 'content': bruto},
                {'role': 'user', 'content': 'O campo args deve ser um objeto JSON (ex.: {"codigo": "..."}).'},
            ]
            continue
    
        chave = indice_arg.get(nome)
        resultado = TOOLS[nome](args.get(chave, '')) if chave else TOOLS[nome]()
        mensagens += [
            {'role': 'assistant', 'content': bruto},
            {'role': 'user', 'content': f'Resultado da ferramenta {nome}:\n{resultado}'},
        ]
    
    return '(limite de passos atingido)'

## O laço em execução

Três perguntas prontas: a primeira roda o caminho clássico
(base -> código -> resposta); as duas seguintes são
intervenções - a última faz o agente chamar wiki_busca
sozinho, sem nota na pergunta.


In [ ]:
# A primeira pergunta segue o caminho clássico: load_csv -> exec_python -> answer
print(run_agent('Quantas reclamacoes tem em cada categoria? Qual a categoria com mais registros?'))

# Comentar/descomentar e reexecutar os trechos abaixo
#print(run_agent('A categoria produto tem mais reclamacoes que a app? Justifique com numeros.'))

# A pergunta abaixo aciona a terceira tool: sem nota extra na pergunta,
# o agente chama wiki_busca sozinho - decidir, agir, observar
#print(run_agent('Pesquise na wikipedia o termo "direito do consumidor" e cite um artigo relacionado.'))
#print(run_agent('Pesquise na wikipedia o termo "Antonio Meneghetti" e cite um artigo relacionado.'))

print('Tokens acumulados:', usage_total())


--- passo 0: {"tool": "wiki_busca", "args": {"termo": "Antonio Meneghetti"}}
    tokens: 291 (prompt 237, completion 54)
--- passo 1: {"tool": "wiki_busca", "args": {"termo": "Ontopsicologia"}}
    tokens: 443 (prompt 295, completion 148)
--- passo 2: Antônio Meneghetti foi o criador da Ontopsicologia. Um artigo relacionado na Wikipédia é **"Ontopsicologia"**: https://pt.wikipedia.org/wiki
    tokens: 465 (prompt 351, completion 114)
--- passo 3: {"answer":"Antônio Meneghetti foi o criador da Ontopsicologia. Um artigo relacionado na Wikipédia é \"Ontopsicologia\": https://pt.wikipedia
    tokens: 562 (prompt 407, completion 155)
Antônio Meneghetti foi o criador da Ontopsicologia. Um artigo relacionado na Wikipédia é "Ontopsicologia": https://pt.wikipedia.org/wiki/Ontopsicologia
Tokens acumulados: {'prompt_tokens': 4968, 'completion_tokens': 1581, 'total_tokens': 6549}


## Critério de parada

Quando um agente para? Neste laço, em quatro níveis:

1. **O modelo decide** - o turno com `{"answer": ...}` termina
   o laço; continuar só acontece quando ele escolhe uma
   `tool`. Quem define 'acabou' é a ação, não o código.
2. **O freio** - `max_steps` existe porque o dev não pode
   confiar no item 1: agente perdido gira para sempre, e o
   teto devolve '(limite de passos atingido)'.
3. **A resposta em si** - o `finish_reason` da API distingue
   `stop` (concluído) de `length` (estourou o orçamento de
   tokens); o `llm.py` levanta erro quando `length`, mesmo
   com conteúdo parcial.
4. **A infraestrutura** - 429 do free tier: até 4 tentativas
   com backoff; insistindo além disso, o erro sobe.

Nos frameworks prontos esta política vem escondida; aqui
dá para ler todas as quatro no arquivo e no `llm.py`.
